**Code to open/merge files on single levels and calculate energy flux potential.**

Code generated by John Cramblitt, borrowing a function to solve the Poisson equation for energy flux potential from John Chiang.

In [ ]:
from netCDF4 import Dataset
import dask
import numpy as np
import numpy.ma as ma
import matplotlib 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import mpl_toolkits
import glob
import pandas as pd
import xarray as xr
import scipy as sp
from scipy import signal
import scipy.stats
import scipy.integrate
import os
import windspharm
import spharm
import _spherepack 
import cftime
from dask.diagnostics import ProgressBar
from pathlib import Path
import re

In [ ]:
## open/merge data files using metadata in file name ##

BASE = Path(r"atm_single_levels")
PREFIX = "atm_monthly_direct_"
SUFFIX = ".nc"
CHUNKS = {"time": 120} 
ENGINE = None                         

rx = re.compile(rf"^{re.escape(PREFIX)}"r"(?:(?P<location>.+?)_VolcanoStart|(?P<no_volcano>NoVolcano)Start)"r"(?P<stamp>\d{8})"r"(?:-e(?P<ensemble>\d+))?"rf"{re.escape(SUFFIX)}$")

def scenario_label(stamp):
    return stamp[-4:]

files_by_loc = {}
for p in BASE.glob(f"{PREFIX}*{SUFFIX}"):
    m = rx.match(p.name)
    if not m:
        continue
    # if it's a NoVolcano file, location="NoVolcano"
    loc = m.group("location") or m.group("no_volcano")
    scen = scenario_label(m.group("stamp"))
    ens  = f"e{m.group('ensemble')}" if m.group("ensemble") else "e0"

    files_by_loc.setdefault(loc, {}).setdefault(scen, {})[ens] = p

SCEN_ORDER = ["1923", "1927"]

def open_location_block(loc, scen_map, chunks=CHUNKS, engine=ENGINE):
    scen_labels = [s for s in SCEN_ORDER if s in scen_map] + [s for s in sorted(scen_map) if s not in SCEN_ORDER]

    scenario_dsets = []
    for scen in scen_labels:
        ens_map = scen_map[scen]
        ens_labels = sorted(ens_map) 
        paths = [ens_map[e] for e in ens_labels]

        ds_ens = xr.open_mfdataset(paths,
                                   combine="nested",
                                   concat_dim="ensemble",
                                   chunks=chunks,
                                   engine=engine,
                                   parallel=True,
                                   decode_times=True,
                                   use_cftime="auto")
        ds_ens = ds_ens.assign_coords(ensemble=ens_labels)
        scenario_dsets.append(ds_ens)

    ds = xr.concat(scenario_dsets, dim="scenario")
    ds = ds.assign_coords(scenario=scen_labels).expand_dims(location=[loc])
    return ds

blocks = [open_location_block(loc, scen_map) for loc, scen_map in sorted(files_by_loc.items())]
ds_all = xr.concat(blocks, dim="location")

ds_all = ds_all.squeeze("lev", drop=True) # drop unnecessary coord
# convert from -180-180 to 0-360
ds_all = ds_all.assign_coords(lon=(((ds_all.lon + 180) % 360) - 180)).sortby("lon")

In [ ]:
# calculate the net energy input (NEI)
# SHFLX -> surface sensible heat flux
# LHFLX -> surface latent heat flux
# FSNT -> net TOA shortwave
# FLNT -> net TOA longwave
# FSNS -> net surface shortwave 
# FLNS -> net surface longwave
ds_all = ds_all.assign(nei=ds_all["SHFLX"] + ds_all["LHFLX"] + ds_all["FSNT"] + ds_all["FLNS"] - ds_all["FLNT"] - ds_all["FSNS"])

In [ ]:
# # Note that there is often an error with using the windspharm package functions, that it expects a certain type of latitude grid.
# # The solution is to regrid the latitude.  See for example https://github.com/ajdawson/windspharm/issues/77

# Create new latitude array 
new_lat_array = np.arange(-90, 91, 180./96)

desiredGrid = ds_all.reindex(lat=new_lat_array, method='nearest')

# Regrid using linear interpolation
ds_regrid = ds_all.interp_like(desiredGrid, method='linear')

In [7]:
## Function to solve the laplacian equation ##

# Inverter Function to get energy flux potential and gradient of energy flux potential

def inverter(dataset, gradient=False, zonal=False):
    
    """Dataset is an xarray dataArray with latitude and longitude coordinates, in that order. The coordinate names should be 'lat' and 'lon.'
    
    gradient, zonal are booleans
    if gradient is True, then the function returns the gradient of energy flux potential
        if zonal is True,  then the function returns the zonal gradient
        if zonal is False, then the function returns the meridional gradient
    otherwise it returns the energy flux potential field"""
    
    HLapData, info = windspharm.tools.prep_data(dataset.data,'yx')  #if the data has a time dimension, add 'tyx' instead of 'yx'
    sph = spharm.Spharmt(dataset.lon.shape[0],dataset.lat.shape[0])
    HlapSpectral = sph.grdtospec(HLapData)
    chiSpectral = _spherepack.invlap(HlapSpectral,6.3712e6)
    chiGrid = sph.spectogrd(chiSpectral)
    chiData = windspharm.tools.recover_data(chiGrid,info)
    HlapInterp = dataset.copy(data=chiData,deep=True)
    if not gradient:
        return HlapInterp
    elif gradient:
        w = windspharm.xarray.VectorWind(HlapInterp,HlapInterp)
        uhHInterp, vhHInterp = w.gradient(HlapInterp)
        uhHInterp, vhHInterp = windspharm.tools.reverse_latdim(uhHInterp, vhHInterp,axis=1)
        if zonal:
            return uhHInterp
        elif not zonal:
            return vhHInterp

In [ ]:
nei = ds_regrid["nei"]  # dims: location, scenario, ensemble, time, lat, lon
efp = xr.full_like(nei, np.nan).load()  
uh  = xr.full_like(nei, np.nan).load()
vh  = xr.full_like(nei, np.nan).load()

# loop through scenarios, locations, ensemble members, and months
for s in ds_regrid.scenario.values:
    for l in ds_regrid.location.values:
        for e in ds_regrid.ensemble.values:
            da = nei.sel(scenario=s, location=l, ensemble=e).dropna("time")

            efp_list, uh_list, vh_list = [], [], []
            for t in da.time.values:
                frame = da.sel(time=t).transpose("lat","lon")
                efp_list.append(inverter(frame, None, None))
                vh_list.append(inverter(frame, True, False).sortby("lon").sortby("lat"))
                uh_list.append(inverter(frame, True, True ).sortby("lon").sortby("lat"))

            efp_sl = xr.concat(efp_list, dim=da.time).transpose("time","lat","lon")
            uh_sl  = xr.concat(uh_list,  dim=da.time).transpose("time","lat","lon")
            vh_sl  = xr.concat(vh_list,  dim=da.time).transpose("time","lat","lon")

            idx = dict(scenario=s, location=l, ensemble=e, time=da.time)
            efp.loc[idx] = efp_sl
            uh.loc[idx]  = uh_sl
            vh.loc[idx]  = vh_sl

ds_regrid = ds_regrid.assign(efp=efp, uh=uh, vh=vh)

In [ ]:
ds_regrid = ds_regrid.drop_vars("time_bnds", errors="ignore") # time_bnds caused errors -> drop
ds_regrid["time"].attrs.pop("bounds", None) # get rid of this

VARS = ["efp","uh","vh","nei","AODVIS","TAUX"] # keep only variables of interest
necessary_vars = ds_regrid[VARS]

ds_regrid.to_netcdf("ALLVARS_allmembers.nc",engine="netcdf4")